In [4]:
import pandas as pd
import numpy as np
import pickle
import re

In [5]:
movies = pickle.load(open("../ml/movies.pkl","rb"))
similarity = pickle.load(open("../ml/similarity.pkl","rb"))
collaborative_model = pickle.load(open("../ml/collaborative_model.pkl","rb"))

In [6]:
ml_movies = pd.read_csv("../dataset/movielens/movies.csv")
links = pd.read_csv("../dataset/movielens/links.csv")

In [7]:
def clean_title(title):
    return re.sub(r"\(\d{4}\)", "", title).strip()

In [8]:
ml_movies["clean_title"] = ml_movies["title"].apply(clean_title)

In [9]:
title_to_movieid = dict(
    zip(
        ml_movies["clean_title"],
        ml_movies["movieId"]
    )
)

In [10]:
print(title_to_movieid.get("Avatar"))

72998


In [11]:
def content_recommend(movie):

    movie_index = movies[movies['title']==movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:21]

    recommendations=[]

    for i in movie_list:
        recommendations.append(
            (
                movies.iloc[i[0]].title,
                i[1]
            )
        )

    return recommendations

In [12]:
def collaborative_score(user_id, movie_title):

    movie_id = title_to_movieid.get(movie_title)

    if movie_id is None:
        return None

    prediction = collaborative_model.predict(user_id,movie_id)

    return prediction.est

In [13]:
CONTENT_WEIGHT = 0.7
COLLAB_WEIGHT = 0.3

In [14]:
def hybrid_recommend(user_id, movie):

    recommendations = []

    content_movies = content_recommend(movie)

    for title, similarity_score in content_movies:

        collab_rating = collaborative_score(user_id, title)

        if collab_rating is None:
            continue

        # Normalize predicted rating (0.5–5 → 0–1)
        normalized_rating = collab_rating / 5

        hybrid_score = (
            CONTENT_WEIGHT * similarity_score +
            COLLAB_WEIGHT * normalized_rating
        )

        recommendations.append(
            (title, hybrid_score, collab_rating)
        )

    recommendations.sort(key=lambda x: x[1], reverse=True)

    return recommendations[:5]

In [15]:
print(hybrid_recommend(1, "Avatar"))

[('Aliens', np.float64(0.47278852331188903), np.float64(4.6991851404949045)), ('Titan A.E.', np.float64(0.42789151724165375), np.float64(4.171134946579241)), ('Predators', np.float64(0.4151934883397078), np.float64(4.028415489264908)), ('Meet Dave', np.float64(0.4144315629934905), np.float64(4.1672095948022205)), ('Battle: Los Angeles', np.float64(0.41250002986929574), np.float64(3.9769851713515827))]


In [16]:
recommendations = hybrid_recommend(1, "Avatar")

for i, movie in enumerate(recommendations, start=1):
    print(f"{i}. {movie[0]}")
    print(f"   Hybrid Score : {movie[1]:.3f}")
    print(f"   Predicted Rating : {movie[2]:.2f}")
    print()

1. Aliens
   Hybrid Score : 0.473
   Predicted Rating : 4.70

2. Titan A.E.
   Hybrid Score : 0.428
   Predicted Rating : 4.17

3. Predators
   Hybrid Score : 0.415
   Predicted Rating : 4.03

4. Meet Dave
   Hybrid Score : 0.414
   Predicted Rating : 4.17

5. Battle: Los Angeles
   Hybrid Score : 0.413
   Predicted Rating : 3.98



## Conclusion

A Hybrid Recommendation System was successfully developed by combining
Content-Based Filtering and Collaborative Filtering.

Content-Based Filtering recommends movies based on metadata such as genres,
keywords, cast, and overview, while Collaborative Filtering predicts user
preferences from historical ratings.

The final recommendations are generated using a weighted hybrid score,
providing more relevant and personalized movie suggestions than either
approach alone.

In [17]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

In [18]:
from ml.hybrid_recommender import hybrid_recommend


In [19]:
recommendations = hybrid_recommend(1, "Avatar")

for i, movie in enumerate(recommendations, start=1):
    print(f"{i}. {movie['title']}")
    print(f"   Similarity Score : {movie['similarity_score']}")
    print(f"   Predicted Rating : {movie['predicted_rating']}")
    print(f"   Hybrid Score     : {movie['hybrid_score']}")
    print()

1. Aliens
   Similarity Score : 0.273
   Predicted Rating : 4.7
   Hybrid Score     : 0.473

2. Titan A.E.
   Similarity Score : 0.254
   Predicted Rating : 4.17
   Hybrid Score     : 0.428

3. Predators
   Similarity Score : 0.248
   Predicted Rating : 4.03
   Hybrid Score     : 0.415

4. Meet Dave
   Similarity Score : 0.235
   Predicted Rating : 4.17
   Hybrid Score     : 0.414

5. Battle: Los Angeles
   Similarity Score : 0.248
   Predicted Rating : 3.98
   Hybrid Score     : 0.413

